<img src='../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

# Python Packaging

In this notebook, you will practice how to organize python code into a package.

The first step will be to set up the basic infrastructure required to run these notebooks. 

**If you are comfortable using git, we recommend checking out a new branch to follow along (`git checkout -b BRANCH_NAME`) during the training.**

You need to:

1. Create a new **package project** with a `pyproject.toml` file in the root of the repository. Run: `uv init --package --name animal_shelter` 
    
2. Install **Polars** as a dependency with `uv add polars`. Notice that this will create a virtual environment and lock file, if it was not already created.

3. Install **Jupyter** as a development dependency so that we can run these notebooks from the virtual environment `uv add --dev jupyter`.

<mark>**Question:**</mark> Why are we adding it as a dev-dependency, but Polars as a normal dependency?

## The use case

Now let's have a look as the use case these training uses as an example. It concerns an animal shelter that is trying to predict the outcome (e.g. adopted, transferred) of the animals that come through it.

In [1]:
import polars as pl
import re

def load_data(path):
    """Load the data and convert the column names.

    Parameters
    ----------
    path : str
        Path to data
    Returns
    -------
    df : polars.DataFrame
        DataFrame with data
    """
    df = pl.read_csv(path, try_parse_dates=True)
    rename_map = {c: convert_camel_case(c.replace("upon", "Upon")) for c in df.columns}
    df = df.rename(rename_map).fill_null("Unknown")
    return df


def convert_camel_case(name):
    """Convert camelCaseString to snake_case_string."""
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()

In [2]:
animal_outcomes = load_data('../data/train.csv')
animal_outcomes.head()

animal_id,name,date_time,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color
str,str,datetime[μs],str,str,str,str,str,str,str
"""A671945""","""Hambone""",2014-02-12 18:22:00,"""Return_to_owner""","""Unknown""","""Dog""","""Neutered Male""","""1 year""","""Shetland Sheepdog Mix""","""Brown/White"""
"""A656520""","""Emily""",2013-10-13 12:44:00,"""Euthanasia""","""Suffering""","""Cat""","""Spayed Female""","""1 year""","""Domestic Shorthair Mix""","""Cream Tabby"""
"""A686464""","""Pearce""",2015-01-31 12:28:00,"""Adoption""","""Foster""","""Dog""","""Neutered Male""","""2 years""","""Pit Bull Mix""","""Blue/White"""
"""A683430""","""Unknown""",2014-07-11 19:09:00,"""Transfer""","""Partner""","""Cat""","""Intact Male""","""3 weeks""","""Domestic Shorthair Mix""","""Blue Cream"""
"""A667013""","""Unknown""",2013-11-15 12:52:00,"""Transfer""","""Partner""","""Dog""","""Neutered Male""","""2 years""","""Lhasa Apso/Miniature Poodle""","""Tan"""


Feel free to spend some time doing some preliminary data exploration:

1. What is the data about?
2. What could be a ML task that we'd like to carry out here? What could be the outcome for predictions?
3. What are the features, and how would you need to transform the data to retrieve those features?
etc.

In [ ]:
# check out the animal_outcomes df if you want here

From this dataset, you can generate the following features about each animal that may be helpful to train a machine learning model later on.

- boolean indicator for whether it is a dog
- boolean indicator for whether it has a name
- categorical feature indicating its sex
- categorical feature indicating whether it is neutered
- catergorical feature indicating its hair type
- age upon outcome in days

All these features are already generated for you with the `add_features(df)` function below:

In [ ]:
import polars as pl


def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add some features to our data.
    Parameters
    ----------
    df : polars.DataFrame
        DataFrame with data (see load_data)
    Returns
    -------
    with_features : polars.DataFrame
        DataFrame with some column features added
    """
    is_dog = check_is_dog(df["animal_type"]).rename("is_dog")

    return df.with_columns(
        is_dog,
        (pl.col("name").str.to_lowercase() != "unknown").alias("has_name"),
        pl.when(pl.col("sex_upon_outcome").str.ends_with("Female"))
          .then(pl.lit("female"))
          .when(pl.col("sex_upon_outcome").str.ends_with("Male"))
          .then(pl.lit("male"))
          .otherwise(pl.lit("unknown"))
          .alias("sex"),
        pl.when(
            pl.col("sex_upon_outcome").str.to_lowercase().str.contains("neutered") |
            pl.col("sex_upon_outcome").str.to_lowercase().str.contains("spayed")
        )
          .then(pl.lit("fixed"))
          .when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("intact"))
          .then(pl.lit("intact"))
          .otherwise(pl.lit("unknown"))
          .alias("neutered"),
        pl.when(pl.col("breed").str.to_lowercase().str.contains("shorthair"))
          .then(pl.lit("shorthair"))
          .when(pl.col("breed").str.to_lowercase().str.contains("medium hair"))
          .then(pl.lit("medium hair"))
          .when(pl.col("breed").str.to_lowercase().str.contains("longhair"))
          .then(pl.lit("longhair"))
          .otherwise(pl.lit("unknown"))
          .alias("hair_type"),
        _compute_days_upon_outcome(pl.col("age_upon_outcome")),
    )


def _compute_days_upon_outcome(age_upon_outcome: pl.Expr) -> pl.Expr:
    period_days = {
        "year": 365, "years": 365, "week": 7, "weeks": 7,
        "month": 30, "months": 30, "day": 1, "days": 1,
    }
    split = age_upon_outcome.str.split(" ")
    time = split.list.first().cast(pl.Float64, strict=False)
    period = (
        split.list.get(1, null_on_oob=True)
        .replace(
            old=list(period_days.keys()),
            new=pl.Series(list(period_days.values()), dtype=pl.Float64),
        )
        .cast(pl.Float64, strict=False)
    )
    return (time * period).alias("days_upon_outcome")


def check_is_dog(animal_type: pl.Series) -> pl.Series:
    """Check if the animal is a dog, otherwise return False.
    Parameters
    ----------
    animal_type : polars.Series
        Type of animal
    Returns
    -------
    result : polars.Series
        Dog or not
    """
    is_cat_dog = animal_type.str.to_lowercase().is_in(["dog", "cat"])
    if not is_cat_dog.all():
        print("Found something else but dogs and cats:\n%s",
              animal_type.filter(~is_cat_dog))
        raise RuntimeError("Found pets that are not dogs or cats.")
    return animal_type.str.to_lowercase() == "dog"

In [ ]:
animal_outcomes = load_data('../data/train.csv')
with_features = add_features(animal_outcomes)
with_features.head()

There are some bad practices going on in the functions above, but don't worry about their quality for now. Let's focus on packaging the code.

## <mark>Exercise:</mark> Successfully import the functions as a package
Your goal is to copy-paste the code from the cells above into a package that exports the functionality that a user (e.g. an analyst writing a report in a notebook or a service serving predictions) would use. 

They should be able to import the functions as in the cell below:

In [ ]:
from animal_shelter.data import load_data
from animal_shelter.features import add_features
animal_outcomes = load_data('../data/test.csv')
with_features = add_features(animal_outcomes)
with_features.head()

***Hints:*** 

1. The location of the package should be in this folder structure: `./src/animal_shelter/__init__.py`
2. You will need two modules `data.py` and `features.py` to successfully run the code above.

You can run the cell below to automatically auto-reload changes to the source code of any imported package, which is very useful during development.

In [ ]:
%load_ext autoreload
%autoreload 2